# 🎙️ Voice Cloning & Speech Synthesis Comprehensive Benchmark
### Evaluating Top Zero-Shot Cloners with [5uck1ess/tts-bench](https://github.com/5uck1ess/tts-bench) Standard Metrics

Models Evaluated:
1. **F5-TTS (CUDA)** — Diffusion Flow-Matching baseline.
2. **Alibaba CosyVoice 2 (`CosyVoice2-0.5B`)** — Qwen2-TTS backbone with zero-shot prompt token conditioning.
3. **Alibaba Qwen3-TTS (`Qwen3-TTS-12Hz-0.6B-Base`)** — Alibaba's newest native zero-shot voice cloning model.
4. **Coqui XTTS-v2** — De facto industry open-source cloning standard (17 languages).
5. **Fish Audio (Fish Speech 1.5 / S2 Pro)** — Dual-AR LLaMA Transformer + Firefly-GAN / DAC codec.
6. **Meta SeamlessM4T v2** — Direct Speech-to-Speech Translation (S2ST) baseline.

---
### Objective Scoring Metrics (tts-bench Standard):
- **SIM (Speaker Similarity)**: WavLM-SV embedding cosine similarity against reference audio ($-1.0$ to $1.0$). Higher = closer voice identity.
- **UTMOS (Naturalness)**: SpeechMOS mean opinion score ($1.0$ to $5.0$). Higher = more human-like.
- **WER (Word Error Rate)**: Whisper ASR transcription error vs target text ($0\%$ to $100\%$). Lower = clearer, no hallucinations.
- **Throughput & RunPod Cost**: Real-Time Factor (RTF) and projected cost for a full 2.4h episode (~80m dialogue).

## ⚙️ Step 1: Dependencies, Reference Audio Sanitizer & Memory Hygiene
**Why sanitizing the reference audio matters:**
The raw `xu_zhisheng_ref_24k_mono.wav` had a different speaker at `0.00–1.10s` and crowd laughter at `3.55–4.40s`.
This step slices `[1.10s : 3.55s]` to isolate **100% pure Xu Zhisheng speech** and sets up clean environment variables.

In [ ]:
# Install core speech packages and tts-bench dependencies
!pip install -q torch torchaudio soundfile librosa transformers accelerate jiwer f5-tts

import os, sys, gc, time, ctypes, types, warnings
import numpy as np
import soundfile as sf
import torch
import IPython.display as ipd

warnings.filterwarnings("ignore", category=FutureWarning)
os.environ["PYTHONHASHSEED"] = "0"
os.environ["COQUI_TOS_AGREED"] = "1"
sys.modules.setdefault("pyworld", types.ModuleType("pyworld"))

RAW_REF_WAV = "xu_zhisheng_ref_24k_mono.wav"
CLEAN_REF_WAV = "xu_zhisheng_clean_pure.wav"
REF_TEXT = "你念完这个我发现我面前正好有条缝让我钻进去。"
GEN_TEXT = "Easy as breathing, they call him the voice of today's workers and the ultimate workhorse, Xu Zhisheng."

def prepare_clean_reference():
    if not os.path.exists(RAW_REF_WAV):
        print(f"⚠️ Please upload '{RAW_REF_WAV}' to Colab root!")
        return None
    data, sr = sf.read(RAW_REF_WAV)
    dur = len(data) / sr
    if dur > 3.5:
        print(f"✂️ Slicing '{RAW_REF_WAV}' ({dur:.2f}s) -> [1.10s : 3.55s] to remove intro speaker & crowd laughter...")
        clean_slice = data[int(1.10 * sr) : int(3.55 * sr)]
        sf.write(CLEAN_REF_WAV, clean_slice, sr)
    else:
        sf.write(CLEAN_REF_WAV, data, sr)
    info = sf.info(CLEAN_REF_WAV)
    print(f"✅ Cleaned Reference: '{CLEAN_REF_WAV}' ({info.duration:.2f}s, {info.samplerate}Hz pure solo speech)")
    return CLEAN_REF_WAV

def free_memory():
    os.environ["PYTHONHASHSEED"] = "0"
    if hasattr(sys, "last_traceback"): sys.last_traceback = None
    if hasattr(sys, "last_value"): sys.last_value = None
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

BENCHMARK_RESULTS = []
GENERATED_FILES = {}

class GPUProfiler:
    def __init__(self, model_name: str):
        self.model_name = model_name
        
    def __enter__(self):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
        self.start_time = time.perf_counter()
        return self
        
    def __exit__(self, exc_type, exc_val, exc_tb):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        self.elapsed = time.perf_counter() - self.start_time
        self.peak_vram_gb = torch.cuda.max_memory_allocated() / (1024**3) if torch.cuda.is_available() else 0.0

    def log_result(self, output_wav_path: str):
        info = sf.info(output_wav_path)
        dur = info.duration
        rtf = dur / self.elapsed if self.elapsed > 0 else 0.0
        
        # Full episode active dialogue ~80 mins = 1.33 hrs
        dialogue_hrs = 1.33
        t4_compute_hrs = dialogue_hrs / rtf if rtf > 0 else 0.0
        cost_a4000 = (t4_compute_hrs / 2.2) * 0.17  # RTX A4000: ~2.2x T4 @ $0.17/hr
        cost_rtx4090 = (t4_compute_hrs / 4.5) * 0.44 # RTX 4090: ~4.5x T4 @ $0.44/hr
        
        result = {
            "Model": self.model_name,
            "Output Audio": output_wav_path,
            "Peak VRAM": f"{self.peak_vram_gb:.2f} GB",
            "T4 Wall Time": f"{self.elapsed:.2f}s",
            "Generated Audio": f"{dur:.2f}s",
            "T4 Speed": f"{rtf:.2f}x RT",
            "RunPod A4000 ($0.17/h)": f"${cost_a4000:.3f}",
            "RunPod RTX 4090 ($0.44/h)": f"${cost_rtx4090:.3f}"
        }
        globals()["BENCHMARK_RESULTS"] = [r for r in globals().get("BENCHMARK_RESULTS", []) if r["Model"] != self.model_name] + [result]
        GENERATED_FILES[self.model_name] = output_wav_path
        print(f"\n📊 [{self.model_name}] COMPUTE REPORT:")
        print(f"  • Peak VRAM     : {self.peak_vram_gb:.2f} GB")
        print(f"  • T4 Speed      : {rtf:.2f}x Real-Time ({self.elapsed:.2f}s compute for {dur:.2f}s audio)")
        print(f"  • RunPod A4000  : ${cost_a4000:.3f} / episode ({dialogue_hrs/(rtf*2.2)*60:.1f} mins compute)")
        print(f"  • RunPod 4090   : ${cost_rtx4090:.3f} / episode ({dialogue_hrs/(rtf*4.5)*60:.1f} mins compute)")

prepare_clean_reference()
if os.path.exists(CLEAN_REF_WAV):
    display(ipd.Audio(CLEAN_REF_WAV))

## 🧪 Model 1: F5-TTS (CUDA Flow-Matching Baseline)
Lightweight (~0.75 GB VRAM) non-autoregressive flow-matching model matching our Apple Silicon baseline.

In [ ]:
from f5_tts.api import F5TTS

free_memory()
OUT_F5 = "output_1_f5_cuda.wav"

if os.path.exists(CLEAN_REF_WAV):
    print("📦 Loading F5-TTS on CUDA...")
    f5 = F5TTS(device="cuda" if torch.cuda.is_available() else "cpu")
    
    with GPUProfiler("F5-TTS (CUDA)") as prof:
        wav, sr, _ = f5.infer(
            ref_file=CLEAN_REF_WAV,
            ref_text=REF_TEXT,
            gen_text=GEN_TEXT,
            nfe_step=32,
            cfg_strength=1.5,
            speed=1.0,
            seed=42
        )
    sf.write(OUT_F5, np.asarray(wav, dtype=np.float32), sr)
    prof.log_result(OUT_F5)
    
    del f5
    free_memory()
    display(ipd.Audio(OUT_F5))

## 🧪 Model 2: Alibaba CosyVoice 2 (`CosyVoice2-0.5B` Zero-Shot)
Alibaba's official `CosyVoice2-0.5B` model utilizing a **Qwen2-TTS** (`Qwen2LM`) speech-token LLM in **zero-shot mode** (retaining prompt acoustic tokens).

In [ ]:
import subprocess, importlib

free_memory()
if not os.path.exists("/content/CosyVoice/cosyvoice"):
    !git clone --depth=1 --recursive https://github.com/FunAudioLLM/CosyVoice.git /content/CosyVoice

cosy_pkgs = [
    "modelscope", "HyperPyYAML", "onnxruntime", "conformer",
    "diffusers", "inflect", "hydra-core", "omegaconf", "openai-whisper",
    "x-transformers", "lightning", "wget", "gdown", "rich", "unidecode", "rootutils"
]
for pkg in cosy_pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], capture_output=True, text=True)
importlib.invalidate_caches()

sys.path.insert(0, "/content/CosyVoice")
sys.path.insert(0, "/content/CosyVoice/third_party/Matcha-TTS")
from modelscope import snapshot_download
from cosyvoice.cli.cosyvoice import CosyVoice2

OUT_COSY2 = "output_2_cosyvoice2.wav"

if os.path.exists(CLEAN_REF_WAV):
    print("📦 Loading Alibaba CosyVoice 2 (CosyVoice2-0.5B / Qwen2-TTS)...")
    model_dir = snapshot_download('iic/CosyVoice2-0.5B', local_dir='/content/CosyVoice/pretrained_models/CosyVoice2-0.5B')
    cosy2 = CosyVoice2(model_dir, load_jit=False, load_trt=False, fp16=False)
    # Unify Qwen2 backbone dtypes to float32 to prevent 'mat1 Float vs mat2 BFloat16' mismatch
    cosy2.model.llm.float()
    
    with GPUProfiler("CosyVoice 2 (0.5B)") as prof:
        # inference_zero_shot retains the prompt speech tokens in the Qwen2 LLM
        output = list(cosy2.inference_zero_shot(GEN_TEXT, REF_TEXT, CLEAN_REF_WAV, stream=False))
        
    chunks = [res["tts_speech"].cpu().float().numpy().squeeze() for res in output]
    speech = np.concatenate(chunks) if len(chunks) > 1 else chunks[0]
    sf.write(OUT_COSY2, speech, cosy2.sample_rate)
    prof.log_result(OUT_COSY2)
    
    del cosy2
    free_memory()
    display(ipd.Audio(OUT_COSY2))

## 🧪 Model 3: Alibaba Qwen3-TTS (`Qwen3-TTS-12Hz-0.6B-Base`)
Alibaba Qwen team's official `Qwen3-TTS` zero-shot voice cloning model (`qwen-tts`).

In [ ]:
import subprocess, importlib

free_memory()
!apt-get install -y sox libsox-dev > /dev/null 2>&1
!pip install -q qwen-tts "huggingface_hub>=1.32.0"
importlib.invalidate_caches()

# Monkey-patch HuggingFace check_model_inputs to support @check_model_inputs() syntax
import transformers.modeling_utils
_orig_check = getattr(transformers.modeling_utils, "check_model_inputs", lambda x: x)
def _patched_check(*args, **kwargs):
    if len(args) == 1 and callable(args[0]):
        return _orig_check(args[0])
    return lambda fn: _orig_check(fn)
transformers.modeling_utils.check_model_inputs = _patched_check

from qwen_tts import Qwen3TTSModel
OUT_QWEN3 = "output_3_qwen3_tts.wav"

if os.path.exists(CLEAN_REF_WAV):
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    print("📦 Loading Qwen3-TTS-12Hz-0.6B-Base on GPU...")
    model_qwen3 = Qwen3TTSModel.from_pretrained(
        "Qwen/Qwen3-TTS-12Hz-0.6B-Base",
        device_map=device,
        dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32
    )
    
    with GPUProfiler("Qwen3-TTS (0.6B-Base)") as prof:
        wavs, sr = model_qwen3.generate_voice_clone(
            text=GEN_TEXT,
            language="English",
            ref_audio=CLEAN_REF_WAV,
            ref_text=REF_TEXT
        )
    speech = wavs[0] if isinstance(wavs, (list, tuple)) else wavs
    if torch.is_tensor(speech): speech = speech.cpu().float().numpy().squeeze()
    sf.write(OUT_QWEN3, np.asarray(speech, dtype=np.float32), sr)
    prof.log_result(OUT_QWEN3)
    
    del model_qwen3
    free_memory()
    display(ipd.Audio(OUT_QWEN3))

## 🧪 Model 4: Coqui XTTS-v2 (`tts_models/multilingual/multi-dataset/xtts_v2`)
Industry zero-shot multilingual cloning baseline (`coqui-tts`, CPML 1.0 license, 17 languages).

In [ ]:
import subprocess, importlib

free_memory()
!pip install -q coqui-tts
importlib.invalidate_caches()

from TTS.api import TTS
OUT_XTTS = "output_4_xtts_v2.wav"

if os.path.exists(CLEAN_REF_WAV):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("📦 Loading Coqui XTTS-v2 on GPU...")
    xtts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
    
    with GPUProfiler("Coqui XTTS-v2") as prof:
        xtts.tts_to_file(
            text=GEN_TEXT,
            speaker_wav=CLEAN_REF_WAV,
            language="en",
            file_path=OUT_XTTS
        )
    prof.log_result(OUT_XTTS)
    
    del xtts
    free_memory()
    display(ipd.Audio(OUT_XTTS))

## 🧪 Model 5: Fish Audio (Fish Speech 1.5 & S2 Pro Architecture)
Dual-AR LLaMA Transformer with Firefly-GAN / DAC acoustic tokenizer.

In [ ]:
import subprocess, importlib, torchaudio

free_memory()
if not os.path.exists("/content/fish-speech"):
    !git clone -b v1.5.0 --depth=1 https://github.com/fishaudio/fish-speech.git /content/fish-speech

fish_pkgs = [
    "einx[torch]==0.2.2", "vector_quantize_pytorch==1.14.24",
    "loralib", "loguru", "pyrootutils", "pydub",
    "opencc-python-reimplemented", "natsort", "ormsgpack", "cachetools"
]
for pkg in fish_pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], capture_output=True, text=True)
importlib.invalidate_caches()

from huggingface_hub import snapshot_download as hf_download
fish_ckpt = "/content/fish-speech/checkpoints/fish-speech-1.5"
if not os.path.exists(f"{fish_ckpt}/model.pth"):
    print("📦 Downloading Fish Speech 1.5 weights from HuggingFace...")
    hf_download(repo_id="fishaudio/fish-speech-1.5", local_dir=fish_ckpt)

sys.path.insert(0, "/content/fish-speech")
from tools.vqgan.inference import load_model as load_vqgan_model
from tools.llama.generate import load_model as load_llama_model, generate_long

OUT_FISH = "output_5_fish_speech.wav"

if os.path.exists(CLEAN_REF_WAV):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    vq_ckpt = f"{fish_ckpt}/firefly-gan-vq-fsq-8x1024-21hz-generator.pth"
    
    print("📦 Loading Fish Speech 1.5 Firefly-GAN & Dual-AR LLaMA...")
    vq_model = load_vqgan_model("firefly_gan_vq", vq_ckpt, device=device)
    llama_model, decode_one_token = load_llama_model(fish_ckpt, device=device, precision=torch.half, compile=False)
    llama_model.setup_caches(max_batch_size=1, max_seq_len=2048, dtype=torch.half)
    
    with GPUProfiler("Fish Speech 1.5") as prof:
        audio, sr = torchaudio.load(CLEAN_REF_WAV)
        if audio.shape[0] > 1: audio = audio.mean(0, keepdim=True)
        audio = torchaudio.functional.resample(audio, sr, vq_model.spec_transform.sample_rate)
        audios = audio[None].to(device)
        audio_lengths = torch.tensor([audios.shape[2]], device=device, dtype=torch.long)
        prompt_tokens = vq_model.encode(audios, audio_lengths)[0][0]
        
        codes_list = []
        for resp in generate_long(
            model=llama_model, device=device, decode_one_token=decode_one_token,
            text=GEN_TEXT, prompt_text=REF_TEXT, prompt_tokens=prompt_tokens,
            top_p=0.7, temperature=0.7, repetition_penalty=1.2
        ):
            if resp.action == "sample":
                codes_list.append(resp.codes)
        
        all_codes = torch.cat(codes_list, dim=1).to(device).long()
        feat_lens = torch.tensor([all_codes.shape[1]], device=device)
        fake_audios, _ = vq_model.decode(indices=all_codes[None], feature_lengths=feat_lens)
        speech = fake_audios[0, 0].float().cpu().numpy()
        
    sf.write(OUT_FISH, speech, vq_model.spec_transform.sample_rate)
    prof.log_result(OUT_FISH)
    
    del vq_model, llama_model
    free_memory()
    display(ipd.Audio(OUT_FISH))

## 🧪 Model 6: Meta SeamlessM4T v2 (Direct S2ST Baseline)
Direct Speech-to-Speech Translation (translates Chinese audio input directly into English speech without generating text intermediate).

In [ ]:
from transformers import AutoProcessor, SeamlessM4Tv2Model
import torchaudio

free_memory()
OUT_SEAMLESS = "output_6_seamless_s2st.wav"

if os.path.exists(CLEAN_REF_WAV):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("📦 Loading Meta SeamlessM4T v2 in FP16...")
    processor = AutoProcessor.from_pretrained("facebook/seamless-m4t-v2-large")
    model_s2st = SeamlessM4Tv2Model.from_pretrained(
        "facebook/seamless-m4t-v2-large",
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True
    ).to(device)
    
    audio, orig_freq = torchaudio.load(CLEAN_REF_WAV)
    if orig_freq != 16000:
        audio = torchaudio.functional.resample(audio, orig_freq, 16000)
    audio_inputs = processor(audio=audio.squeeze().numpy(), return_tensors="pt", sampling_rate=16000).to(device, torch.float16)
    
    with GPUProfiler("SeamlessM4T v2 (S2ST)") as prof:
        with torch.no_grad():
            out_audio = model_s2st.generate(**audio_inputs, tgt_lang="eng", generate_speech=True)
            
    speech = out_audio[0].cpu().float().numpy().squeeze()
    sample_rate = getattr(model_s2st.config, 'sampling_rate', 16000)
    sf.write(OUT_SEAMLESS, speech, sample_rate)
    prof.log_result(OUT_SEAMLESS)
    
    del model_s2st, processor, audio_inputs, out_audio
    free_memory()
    display(ipd.Audio(OUT_SEAMLESS))

## 🏆 Step 7: Automated [5uck1ess/tts-bench](https://github.com/5uck1ess/tts-bench) Evaluation Suite
Scores every generated clip across the canonical TTS benchmark metrics:
1. **`SIM` (Speaker Similarity)**: WavLM-SV embedding cosine similarity vs. the pure Xu Zhisheng reference clip.
2. **`UTMOS` (Naturalness)**: Reference-free Mean Opinion Score predictor via `SpeechMOS utmos22_strong` ($1.0$ to $5.0$).
3. **`WER` (Word Error Rate)**: Whisper ASR vs intended target English text (catches dropped words, slurs, or hallucinations like *"funny oh yeah"*).
4. **`Overall Quality Score`**: Composite metric $\text{Score} = (\text{SIM} \times 40) + (\frac{\text{UTMOS}}{5.0} \times 40) + ((1 - \text{WER}) \times 20)$.

In [ ]:
import torch, librosa, jiwer, re
import torch.nn.functional as F
import pandas as pd
from transformers import AutoFeatureExtractor, AutoModelForAudioXVector, pipeline
from IPython.display import display

free_memory()
device_id = 0 if torch.cuda.is_available() else -1
device_str = "cuda" if torch.cuda.is_available() else "cpu"

print("📥 Loading tts-bench Scorers...")
# 1. WavLM Speaker Verification
wavlm_fe = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base-plus-sv")
wavlm_model = AutoModelForAudioXVector.from_pretrained("microsoft/wavlm-base-plus-sv").to(device_str).eval()

def get_speaker_emb(wav_path):
    y, _ = librosa.load(wav_path, sr=16000, mono=True)
    inp = wavlm_fe(y, return_tensors="pt", sampling_rate=16000).to(device_str)
    with torch.no_grad():
        emb = wavlm_model(**inp).embeddings
    return F.normalize(emb, dim=-1)

# 2. UTMOS Naturalness Predictor (SpeechMOS)
try:
    utmos_model = torch.hub.load("tarepan/SpeechMOS:v1.0.0", "utmos22_strong", trust_repo=True).to(device_str).eval()
except Exception as e:
    print(f"⚠️ UTMOS load warning: {e}")
    utmos_model = None

# 3. Whisper ASR for Intelligibility / WER
asr_pipe = pipeline("automatic-speech-recognition", model="openai/whisper-medium", device=device_id)

def clean_txt(t):
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", " ", t.lower())).strip()

ref_emb = get_speaker_emb(CLEAN_REF_WAV)
intended_norm = clean_txt(GEN_TEXT)

SCORED_ROWS = []
print(f"\n🏆 RUNNING TTS-BENCH EVALUATION ON {len(GENERATED_FILES)} CLIPS...\n")

for res in BENCHMARK_RESULTS:
    m_name = res["Model"]
    f_path = res["Output Audio"]
    if not os.path.exists(f_path):
        continue
        
    # A. SIM (Speaker Cosine Similarity)
    gen_emb = get_speaker_emb(f_path)
    sim = float(torch.cosine_similarity(ref_emb, gen_emb).item())
    
    # B. UTMOS (Naturalness MOS 1.0 - 5.0)
    utmos = 0.0
    if utmos_model is not None:
        y_16k, _ = librosa.load(f_path, sr=16000, mono=True)
        x_t = torch.from_numpy(np.asarray(y_16k, dtype=np.float32)).unsqueeze(0).to(device_str)
        with torch.no_grad():
            utmos = float(utmos_model(x_t, 16000).item())
            
    # C. WER (Intelligibility)
    asr_out = asr_pipe(f_path)["text"]
    wer = float(jiwer.wer(intended_norm, clean_txt(asr_out)))
    
    # D. Composite Overall Score (out of 100)
    # 40% Voice Match (SIM), 40% Naturalness (UTMOS), 20% Accuracy (1 - WER)
    quality_score = (max(0, sim) * 40.0) + ((min(5.0, utmos) / 5.0) * 40.0) + (max(0.0, 1.0 - wer) * 20.0)
    
    SCORED_ROWS.append({
        "Model": m_name,
        "Quality Score (0-100)": f"{quality_score:.1f}",
        "SIM (Voice Match)": f"{sim:.4f}",
        "UTMOS (MOS 1-5)": f"{utmos:.2f}",
        "WER (Error %)": f"{wer*100:.1f}%",
        "T4 Speed": res["T4 Speed"],
        "RunPod A4000 Cost": res["RunPod A4000 ($0.17/h)"],
        "RunPod RTX 4090 Cost": res["RunPod RTX 4090 ($0.44/h)"],
        "Whisper Transcription": asr_out
    })

df_scores = pd.DataFrame(SCORED_ROWS).sort_values(by="Quality Score (0-100)", ascending=False).reset_index(drop=True)
print("=" * 95)
print("🏅 FINAL TTS-BENCH LEADERBOARD (RANKED BY OVERALL QUALITY SCORE)")
print("=" * 95)
display(df_scores)

del wavlm_model, asr_pipe
if utmos_model is not None: del utmos_model
free_memory()